In [2]:
import os
import json 
import regex as re
from tqdm import tqdm
import pandas as pd
from typing import List, Dict
from IPython.core.display import HTML
from atlassian import Confluence
import html2text


pd.set_option('display.max_columns', None)


url = 'https://innovaccer.atlassian.net'
username = os.getenv("ATLASSIAN_USERNAME")
password = os.getenv("ATLASSIAN_PASSWORD")

In [3]:
class ConfluenceParser:
    def __init__(self, space: str) -> None:
        self.space = space
        self.confluence = Confluence(
            url=url,
            username=username,
            password=password,
        )
        
    def _html_to_markdown(self, html_string: str):
        """ Converts a markdown string to plaintext """
        return html2text.html2text(html_string)
    
    def render_html(self, markdown_text: str):
        return HTML(markdown_text)

    def _extract_href_from_text(self, string: str) -> List:
        """
        Find all hrefs in the markdown text
        """
        return re.findall(r'href="([^"]+)"', string)

    def get_all_page_ids_from_space(self) -> List:
        """
        Get the list of all page-ids from a Confluence Space
        """
        max_pages = 100000
        limit = 100
        all_page_ids = []

        for i in range(0,max_pages,limit):
            pages = self.confluence.get_all_pages_from_space(
                space=self.space,
                start=i,
                limit=limit
            )
            page_ids = [page["id"] for page in pages if page["status"]=='current']
            if len(page_ids) == 0:
                break
            else:
                all_page_ids.extend(page_ids)
        
        return all_page_ids


    def get_content_for_single_page(self, page_id: str) -> Dict:
        """
        Get the page body and other metadata from a single page.
        """
        # get url for page-id
        url = f"{self.confluence.url}/spaces/{self.space}/pages/{page_id}"

        # get page body
        page_contents = self.confluence.get_page_by_id(page_id, expand='body.storage', status=None, version=None)
        html = page_contents['body']['storage']['value']
        # body = self._html_to_markdown(html)
        body = self._html_to_markdown(page_contents['body']['storage']['value']) # confluence_arb[110]['content']['markdown'])
        hrefs = self._extract_href_from_text(html)
        title = page_contents["title"]

        # get page ancestors
        ancestors_contents = self.confluence.get_page_by_id(page_id, expand='ancestors', status=None, version=None)
        ancestors = {item["id"]:item["title"]  for item in ancestors_contents['ancestors']}

        # compile content
        content = {
            "url": url,
            "content": {
                "space": self.space,
                "page_id": page_id,
                "title": title,
                "body": body,
                "html": html,
                "hrefs": hrefs,
                "ancestors": ancestors
            }
        }

        return content

    def get_content_for_all_pages(self) -> List:
        """
        Get the page body and other metadata from all pages.
        """
        contents = []

        # get all page-ids from space
        page_ids = self.get_all_page_ids_from_space()

        # read body and metadata of all pages within the confluence space
        for page_id in tqdm(page_ids):
            content = self.get_content_for_single_page(page_id)
            contents.append(content)
        
        return contents


In [4]:
space = "PO"

confluence_parser = ConfluenceParser(space=space)
contents_arb = confluence_parser.get_content_for_all_pages()

# save confluence contents as JSON
json.dump(contents_arb, open(f"../src/data/confluence_{space.lower()}.json", "w"))

100%|██████████| 254/254 [01:33<00:00,  2.72it/s]
